# YOLO11m 학습 노트북

알약 사진에서 **알약 위치(bbox)와 약 종류(56종)**를 함께 찾는 모델을 YOLO11m으로 학습한다.
위에서 아래로 실행하면 **전처리 → 학습 → 검증 → 오류 분석**까지 진행된다.

## 왜 YOLO11인가 (참고 논문 요약)

참고: *YOLO11 vs AutoML 알약 검출 비교* (PMC13068000)

| 논문에서 확인한 점 | 우리 프로젝트에 주는 의미 |
|---|---|
| YOLO11은 **1단계 검출기**: 사진 전체를 한 번 순전파해서 박스와 클래스 확률을 함께 예측 | 사진 한 장에서 알약 2~4개의 위치와 종류를 동시에 찾는 우리 과제에 그대로 맞음 |
| 코드 기반이라 **모델 구조와 학습 파라미터를 직접 제어**할 수 있음(AutoML은 맞춤화가 제한적) | 분할 규칙, 증강, NMS를 우리 데이터에 맞게 정할 수 있음 |
| **플랫폼 비용 0**, 엣지 배포에 유리 | Colab이나 개인 GPU만으로 실험 가능 |
| 학습 데이터가 늘수록 **성능이 가장 일관되게, 거의 선형으로 오름** | 나중에 외부 데이터를 넣으면 효과를 기대할 수 있음 |
| 가장 까다로운 다중 알약 데이터셋에서 향상 폭이 가장 컸음(재현율 0.49 → 0.91) | 우리처럼 한 사진에 여러 알약이 있는 경우에도 데이터를 늘리면 개선 여지가 큼 |
| 대신 최적 성능에는 **더 큰 학습 데이터**와 학습 시간이 필요함 | 우리 데이터(232장)만으로는 한계가 있음 → 외부 데이터 계획과 연결됨 |

## 왜 m(medium) 크기인가

논문도 **속도·정확도·학습 시간의 균형**을 이유로 YOLO11m을 썼다. 우리 상황에서도 같은 판단이 맞는다.

| 기준 | 우리 상황 | YOLO11m이 맞는 이유 |
|---|---|---|
| **어려운 부분** | 위치 찾기는 쉬움(배경이 균일). **어려운 건 약 종류 구분**: 색·모양이 비슷하고 각인으로만 갈리는 약이 많음 | n·s보다 표현력이 커서 각인이나 색조 같은 미세한 차이를 구분하기 유리함 |
| **데이터 양** | 지금 232장, 나중에 외부 데이터 추가 예정 | COCO 사전학습 가중치에서 시작(전이학습)해 적은 데이터로도 학습 가능. 데이터가 늘면 큰 모델일수록 이득이 큼. 나중에 ai_hub에서 추가 데이터를 받을 수 있기에 이러한 면에서도 적합 |
| **알약 크기** | 원본 976×1280에서 약 150~450px, 640으로 줄여도 약 100~300px | 작은 물체가 아니라 기본 입력 640으로 충분 |
| **학습 환경** | Colab T4(16GB) 이상 | 파라미터 약 2천만 개로 T4에서 batch 16 학습 가능 |

| 모델 | 파라미터 | COCO mAP50-95 | 이 프로젝트에서 |
|---|---:|---:|---|
| YOLO11n | 2.6M | 39.5 | 가장 빠름. 비교 기준선 |
| YOLO11s | 9.4M | 47.0 | 중간 |
| **YOLO11m** | **20.1M** | **51.5** | **선택**: Colab GPU로 감당할 수 있는 범위에서 구분 능력이 크게 오름 |
| YOLO11l | 25.3M | 53.4 | m 대비 이득은 작고 학습은 느림 |

### 주의할 점

- **과적합 위험**: 232장만으로는 큰 모델이 사진을 외울 수 있다. 대응책은 세 가지다.
  ① 세팅 묶음 단위 분할로 val 점수가 부풀려지지 않게 한다.
  ② 증강으로 장면 암기를 줄인다.
  ③ 같은 분할에서 n/s도 학습해 비교한다(`CFG["weights"]`만 바꾸면 된다).
- **학습 시간**: 논문 말대로 m은 학습이 오래 걸린다. 동작 확인은 `epochs=1`로 짧게 하고, 전체 학습은 GPU에서 돌린다.
- **팀 문서**: `AGENTS.md`에는 아직 "검출 모델 미정"으로 되어 있다. 확정되면 문서도 갱신한다.

## 전체 흐름

```
원본 json (data/raw)                         외부 데이터 (나중에, 선택)
      │                                            │
      ▼                                            ▼
[2] 박스 표로 읽기 + EDA 보완 반영            [4] 박스 표로 읽기
      ▼                                            │
[3] 세팅 묶음 → train / val 분할                   ▼
      │                                  누수 필터 (val과 같은 장면 제거)
      └──────────────┬─────────────────────────────┘
                     ▼   val = 우리 데이터만 / train = 우리 train + 외부
[5] YOLO 형식 변환: images/, labels/(.txt), data.yaml
                     ▼
[6] YOLO11m 학습 (학습 루프·증강은 ultralytics가 실행)
                     ▼
[7] 표준 지표: mAP50, mAP50-95, 약별 AP
[8] 오류 양상 (논문 방식, conf 0.5): 정밀도·재현율·FNR·OER + 착각 표
[9] (선택) 데이터 양별 성능 곡선: 외부 데이터를 넣을 가치가 있는지 판단
```

**핵심은 "같은 표 형식"이다.** 우리 데이터든 외부 데이터든 먼저 `한 행 = 박스 하나` 표로 바꾸면,
그 뒤 단계(분할, 변환, 학습)는 출처와 관계없이 똑같이 돌아간다. 외부 데이터를 붙일 때 고칠 곳은 1번 설정의 경로뿐이다.

| 규칙 | 이유 |
|---|---|
| 같은 **세팅 묶음**은 train과 val 중 한쪽에만 | 형제 사진이 양쪽에 있으면 점수가 부풀려짐 |
| 박스 5개 이하인 **희귀한 약**의 묶음은 train 고정 | val로 보내면 배울 데이터가 사라짐 |
| **val은 우리 데이터만**, 외부 데이터는 train에만 | 대회 test와 같은 조건에서 평가 |
| **증강은 train에만** | val은 실제 사진 그대로 평가 |

## 0. 환경 준비 (Colab)

Colab에서는 처음에 `/content`에서 시작하므로, 저장소 폴더(`/content/pill-bbox`)로 옮기고 패키지를 맞춘다.
로컬에서 실행하면 이 셀은 아무것도 하지 않는다.

- `data/raw/`와 `data/processed/`는 git에 올라가지 않는다. Colab에 데이터가 없으면 Drive의 `pill_data.zip`을 풀어서 채운다.
- zip 만들기(로컬 터미널): `zip -r pill_data.zip data/raw/train_images data/raw/train_annotations data/processed/added_annotations.csv data/processed/fixed_annotations.csv`

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COLAB_ROOT = Path("/content/pill-bbox")
DATA_ZIP = Path("/content/drive/MyDrive/pill_data.zip")  # Drive에 올린 위치가 다르면 여기만 수정


def run(cmd: list[str]) -> None:
    """셸 명령을 실행하고 출력을 셀에 보여 준다."""
    out = subprocess.run(cmd, capture_output=True, text=True, check=True)
    print(out.stdout + out.stderr)


if IN_COLAB:
    os.chdir(COLAB_ROOT)  # 이후 셀은 저장소 폴더에서 실행된다
    run(["bash", "scripts/setup.sh"])
    if not (COLAB_ROOT / "data" / "raw" / "train_images").exists():
        from google.colab import drive

        drive.mount("/content/drive")
        run(["unzip", "-q", "-o", str(DATA_ZIP), "-d", str(COLAB_ROOT)])
print("작업 위치:", Path.cwd())

### 🔍 코드 풀이: 0. 환경 준비

> 비유: **작업실에 들어가서 → 도구를 챙기고 → 창고에서 재료를 꺼내 오는** 단계

| 코드 | 쉽게 말하면 |
|---|---|
| `IN_COLAB = "google.colab" in sys.modules` | "지금 Colab에서 돌고 있나?"를 확인하는 스위치. Colab이면 `True` |
| `def run(cmd)` | 터미널 명령을 대신 쳐 주고, 결과를 셀 아래에 보여 주는 **심부름꾼** 함수 |
| `os.chdir(COLAB_ROOT)` | 작업실(`/content/pill-bbox`) 문을 열고 들어간다. 터미널의 `cd`와 같음 |
| `run(["bash", "scripts/setup.sh"])` | 팀이 정한 **장보기 목록**(`uv.lock`)대로 패키지를 맞추고, GPU가 잡혔는지 알려 준다 |
| `if not (...).exists():` | 재료(사진)가 **없을 때만** 아래를 실행. 이미 있으면 건너뜀 |
| `drive.mount(...)` | Google Drive라는 **창고**를 Colab에 연결 (처음엔 로그인 창이 뜸) |
| `run(["unzip", ...])` | 창고에서 꺼낸 **택배 상자**(`pill_data.zip`)를 작업실에 푼다 |

마지막 줄이 `작업 위치: /content/pill-bbox`로 나오면 성공이다.

## 1. 설정

설정을 **바뀌었을 때 다시 실행해야 하는 범위**에 따라 세 묶음으로 나눈다.

| 묶음 | 내용 | 바꾸면 |
|---|---|---|
| `DATA_CFG` | train/val 분할 규칙 (fold, 세팅 묶음 기준, 희귀 약 기준) | 3~5단계(분할·변환)부터 다시 실행 |
| `TRAIN_CFG` | 모델과 학습 하이퍼파라미터, 증강 | 6단계(학습)만 다시 실행 |
| `EVAL_CFG` | 오류 분석 기준 (논문 방식 conf 0.5, IoU 0.5) | 8단계만 다시 실행 |

`TRAIN_CFG`는 **모든 실험의 기본값**이다. 실험마다 다른 값은 여기를 고치지 않고, 6단계에서 **바꿀 값만** 적어 덮어쓴다.
키 이름은 ultralytics `model.train()`의 인자 이름과 같게 맞춰 두었기 때문에, 새 하이퍼파라미터도 키 하나만 추가하면 된다.
외부 데이터를 쓸 때는 `EXTERNAL_ANN_DIR`, `EXTERNAL_IMG_DIR`만 채운다.

In [ ]:
import sys
from pathlib import Path

# 노트북이 어디서 실행되든 프로젝트 루트(pyproject.toml이 있는 폴더)를 찾는다 (로컬·Colab 공통)
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        f"프로젝트 루트를 못 찾았다 (지금 위치: {Path.cwd()}). 0번 환경 준비 셀을 먼저 실행한다."
    )
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TRAIN_IMG_DIR = RAW_DIR / "train_images"
ANN_DIR = RAW_DIR / "train_annotations"
ADDED_PATH = PROCESSED_DIR / "added_annotations.csv"  # EDA에서 새로 추가한 박스
FIXED_PATH = PROCESSED_DIR / "fixed_annotations.csv"  # EDA에서 위치를 고친 박스
RUNS_DIR = PROJECT_ROOT / "runs"  # 학습 결과와 가장 좋은 가중치를 모아 두는 곳 (git 제외)

SEED = 42  # 분할과 학습 모두 이 값으로 고정해 같은 결과를 다시 만들 수 있게 한다

# 데이터·분할 설정: 바꾸면 3~5단계부터 다시 실행
DATA_CFG = {
    "fold": 0,  # 0 ~ n_splits-1. 이번 실험의 val로 쓸 묶음 번호
    "n_splits": 5,
    "same_dist": 30,  # 세팅 공유 판정: 박스 중심 거리(px)
    "min_same": 2,  # 세팅 공유 판정: 같은 자리여야 하는 공통 약 수
    "rare_max": 5,  # 박스가 이 개수 이하인 약 → 그 묶음은 train 고정
}

# 학습 기본값: 모든 실험의 출발점. 키 이름 = ultralytics train() 인자 이름
TRAIN_CFG = {
    "weights": "yolo11m.pt",  # 시작 가중치. 크기 비교는 yolo11n.pt / yolo11s.pt
    "epochs": 100,
    "imgsz": 640,
    "batch": 16,  # T4 기준. -1이면 GPU 메모리에 맞춰 자동
    "patience": 30,  # val 성능이 이만큼 epoch 동안 안 오르면 조기 종료
    "optimizer": "auto",  # "auto"면 lr0를 무시하고 자동 선택. lr0를 실험하려면 "AdamW" 등으로
    "lr0": 0.01,
    "seed": SEED,
    # 증강 (train에만 적용)
    "hsv_h": 0.005,  # 색조: 색이 약 구분 단서라 아주 약하게
    "hsv_s": 0.5,
    "hsv_v": 0.3,  # 밝기: 가장자리 조명 얼룩 대비
    "degrees": 10.0,  # 회전
    "flipud": 0.5,
    "fliplr": 0.5,
    "mosaic": 1.0,
    "close_mosaic": 10,  # 마지막 10 epoch은 mosaic 끔
}

# 오류 분석 기준 (논문 관례)
EVAL_CFG = {
    "conf": 0.5,  # 이 신뢰도 이상인 예측만 확정
    "iou": 0.5,  # 정답 박스와 이만큼 겹치면 같은 위치로 봄
}

# 외부 데이터 (쓰지 않으면 None). 우리 데이터와 같은 COCO 형식 json이라고 가정한다
EXTERNAL_ANN_DIR = None  # 예: Path("/content/external/annotations")
EXTERNAL_IMG_DIR = None  # 예: Path("/content/external/images")
EXTERNAL_NEW_CLASSES = False  # False면 우리 56종에 없는 약은 외부 데이터에서 뺀다

YOLO_DIR = PROCESSED_DIR / "yolo" / f"fold{DATA_CFG['fold']}"
print("프로젝트 루트:", PROJECT_ROOT)
print(f"기본 모델: {TRAIN_CFG['weights']} · fold {DATA_CFG['fold']} · seed {SEED}")
print(
    f"데이터: train 사진 {len(list(TRAIN_IMG_DIR.glob('*.png')))}장 · "
    f"json 폴더 {'있음' if ANN_DIR.exists() else '없음'} · "
    f"EDA 보완 csv {'있음' if ADDED_PATH.exists() and FIXED_PATH.exists() else '없음'}"
)

### 🔍 코드 풀이: 1. 설정

> 비유: 요리 전에 **레시피 카드**를 적어 두는 단계. 카드를 세 장으로 나눴다: **장보기 카드**(데이터), **조리법 카드**(학습), **맛 평가 카드**(평가).

**① 프로젝트 루트 찾기**

```python
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if ...), None)
```

지금 폴더에서 **계단을 한 칸씩 올라가며** `pyproject.toml`이라는 **문패**가 붙은 폴더를 찾는다.
끝까지 못 찾으면 `None` → "0번 셀을 먼저 실행하라"는 안내를 띄운다.

**② 경로 만들기**

`PROJECT_ROOT / "data" / "raw"`처럼 `/`로 **주소를 이어 붙인다**. 윈도우·맥·리눅스 어디서든 알맞게 바뀐다.

**③ 카드 세 장**

| 카드 | 비유 | 바꾸면 |
|---|---|---|
| `DATA_CFG` | **장보기 카드**: 어떤 재료를 연습용(train)과 시험용(val)으로 나눌지 | 재료를 다시 나눠야 하니 3단계부터 |
| `TRAIN_CFG` | **조리법 카드**: 불 세기, 조리 시간, 양념 | 요리(학습)만 다시 |
| `EVAL_CFG` | **맛 평가 카드**: 몇 점 이상을 "맛있다"로 칠지 | 평가만 다시 |

**④ 조리법 카드(`TRAIN_CFG`)에서 자주 바꾸는 것**

| 키 | 뜻 | 비유 |
|---|---|---|
| `weights` | 어떤 모델로 시작할지 | 어떤 **선수**를 데려올지 (n·s·m) |
| `epochs` | 전체 사진을 몇 바퀴 공부할지 | 문제집을 **몇 회독** 할지 |
| `batch` | 한 번에 몇 장씩 볼지 | 한 번에 **몇 문제씩** 풀고 채점할지 |
| `patience` | 몇 바퀴 동안 안 늘면 멈출지 | 성적이 **30회 연속** 안 오르면 그만 |
| `lr0` | 한 번에 얼마나 크게 고칠지 | 오답 노트를 보고 **습관을 얼마나 확 바꿀지** |
| `hsv_h`, `degrees`, `mosaic` … | 증강 | 같은 문제를 **조금씩 바꿔 낸 변형 문제** |

> ⚠️ `optimizer`가 `"auto"`면 ultralytics가 학습률을 알아서 정하기 때문에 `lr0`를 바꿔도 **효과가 없다**. `lr0`를 실험할 때는 `optimizer`도 `"AdamW"`나 `"SGD"`로 정한다.

**⑤ 이렇게 나눈 이유**

이 카드는 **기본값**이다. 실험할 때는 카드를 고쳐 쓰지 않고, 6단계에서 **"이번엔 이것만 다르게"**라고 메모만 붙인다.

```python
{"name": "lr_low", "optimizer": "AdamW", "lr0": 0.001}  # 나머지는 기본값 그대로
```

키 이름이 ultralytics와 같아서, 새 하이퍼파라미터도 **키 하나만 추가**하면 된다.

**⑥ 마지막 `print`**

요리 전 **재료 점검**: 사진 232장, json 폴더, EDA 보완 csv가 다 있는지 확인한다.

## 2. 박스 표로 읽기

모든 박스를 **한 행 = 박스 하나**인 표(DataFrame)로 바꾼다. 이후 단계(분할, 변환, 학습)는 이 표만 보고 동작한다.

```
json 파일 (사진 1장 + 박스 여러 개)          박스 표
┌───────────────────────────┐          file_name | category_id | x | y | w | h | ...
│ images: [{file_name, ...}]│   →      사진A.png |   3351      | … | … | … | …
│ annotations: [{bbox}, ...]│          사진A.png |  16262      | … | … | … | …
└───────────────────────────┘          사진B.png |   3351      | … | … | … | …
```

- **왜 표로 바꾸나**: 외부 데이터도 같은 표 모양으로 바꾸면, 뒤 단계를 출처와 관계없이 그대로 쓸 수 있다.
- `img_path`: 실제 이미지 위치 / `source`: `ours`(우리 데이터) 또는 `external`(외부 데이터)
- `x, y, w, h`: 박스 왼쪽 위 좌표와 너비·높이 (픽셀, COCO 형식)
- **EDA 보완 반영**: 원본 json은 건드리지 않고, 읽을 때만 위치 수정(`fixed`)과 추가 박스(`added`)를 합친다.
- 정상이라면 **사진 232장 · 약 56종**, EDA 보완은 둘 다 **반영**으로 나온다.

In [ ]:
import json

import pandas as pd

BOX_COLS = ["file_name", "img_path", "source", "category_id", "x", "y", "w", "h", "width", "height"]


def load_coco_boxes(ann_dir: Path, img_dir: Path, source: str) -> pd.DataFrame:
    """COCO 형식 json 폴더를 박스 표로 읽는다. 이미지 파일이 없는 박스는 버린다."""
    rows = []
    for path in sorted(ann_dir.rglob("*.json")):
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        images = {img["id"]: img for img in data["images"]}
        for ann in data["annotations"]:
            img = images[ann["image_id"]]
            x, y, w, h = ann["bbox"]
            rows.append(
                {
                    "file_name": img["file_name"],
                    "img_path": str(img_dir / img["file_name"]),
                    "source": source,
                    "category_id": ann["category_id"],
                    "x": x,
                    "y": y,
                    "w": w,
                    "h": h,
                    "width": img["width"],
                    "height": img["height"],
                }
            )
    boxes = pd.DataFrame(rows, columns=BOX_COLS)
    exists = {p: Path(p).exists() for p in boxes["img_path"].unique()}
    return boxes[boxes["img_path"].map(exists)].reset_index(drop=True)


def load_our_boxes() -> pd.DataFrame:
    """우리 train 박스 + EDA 보완 (위치 수정 → 추가 박스 순서로 반영)."""
    boxes = load_coco_boxes(ANN_DIR, TRAIN_IMG_DIR, source="ours")
    meta = boxes.drop_duplicates("file_name").set_index("file_name")[
        ["img_path", "source", "width", "height"]
    ]
    if FIXED_PATH.exists():
        fixes = pd.read_csv(FIXED_PATH).set_index(["file_name", "category_id"])[
            ["x", "y", "w", "h"]
        ]
        key = pd.MultiIndex.from_frame(boxes[["file_name", "category_id"]])
        target = key.isin(fixes.index)
        boxes.loc[target, ["x", "y", "w", "h"]] = fixes.loc[key[target]].values
    if ADDED_PATH.exists():
        added = pd.read_csv(ADDED_PATH)[["file_name", "category_id", "x", "y", "w", "h"]]
        boxes = pd.concat([boxes, added.join(meta, on="file_name")], ignore_index=True)
    return boxes[BOX_COLS].reset_index(drop=True)


ours = load_our_boxes()
print(
    f"우리 데이터: 사진 {ours['file_name'].nunique()}장 · 박스 {len(ours)}개 · "
    f"약 {ours['category_id'].nunique()}종"
)
print(
    f"EDA 보완: 위치 수정 {'반영' if FIXED_PATH.exists() else '파일 없음'}, "
    f"추가 박스 {'반영' if ADDED_PATH.exists() else '파일 없음'}"
)
ours.head()

### 🔍 코드 풀이: 2. 박스 표로 읽기

> 비유: 흩어진 **영수증(json)** 수백 장을 **엑셀 한 장(표)**으로 옮겨 적는 단계

**① `load_coco_boxes`: json → 표**

1. json 파일을 **한 장씩 연다** (`for path in ...rglob("*.json")`)
2. `images = {img["id"]: img ...}`: 사진 정보를 **번호표 사물함**에 넣어 둔다. 박스의 `image_id`만 보면 어느 사진인지 바로 꺼낼 수 있다.
3. 박스(`annotations`)마다 **한 줄씩** `rows`에 적는다 (사진 이름, 약 코드, 좌표 x·y·w·h)
4. `pd.DataFrame(rows)`: 적은 줄들을 **표**로 만든다
5. 마지막 두 줄: 사진 파일이 실제로 없는 줄은 **지운다** (영수증만 있고 물건이 없는 경우)

**② `load_our_boxes`: 원본 + EDA 보완**

| 단계 | 하는 일 | 비유 |
|---|---|---|
| `load_coco_boxes(...)` | 원본 박스를 읽음 | 원본 영수증 |
| `if FIXED_PATH.exists():` | 같은 사진·같은 약의 박스를 찾아 **좌표를 바꿔 끼움** | 틀린 칸에 **수정테이프** |
| `if ADDED_PATH.exists():` | 빠졌던 박스를 표 **아래에 붙임**. 사진 크기 같은 정보는 `meta`에서 채움 | 빠진 영수증 **추가 첨부** |

원본 json 파일 자체는 **건드리지 않는다**. 읽을 때만 고쳐서 쓴다.

**③ 마지막 출력**

- `ours['file_name'].nunique()`: 사진 이름의 **중복 없는 개수** = 사진 수
- `len(ours)`: 표의 줄 수 = 박스 수
- `ours.head()`: 표의 **첫 5줄 미리보기**